In [2]:
import pandas as pd 

df = pd.read_csv("APL_Logistics.csv", encoding="latin1" )
df.head() 

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Product Name,Product Price,Shipping Mode
0,DEBIT,6,4,159.69,472.45,Late delivery,1,9,Cardio Equipment,Brownsville,...,5,499.95,472.45,159.69,South Asia,Maharashtra,COMPLETE,Nike Men's Free 5.0+ Running Shoe,99.99,Standard Class
1,DEBIT,4,4,48.71,167.96,Shipping on time,0,29,Shop By Sport,Littleton,...,5,199.95,167.96,48.71,Central America,Cortés,ON_HOLD,Under Armour Girls' Toddler Spine Surge Runni,39.99,Standard Class
2,DEBIT,4,4,87.36,181.99,Shipping on time,0,48,Water Sports,Littleton,...,1,199.99,181.99,87.36,Central America,Cortés,ON_HOLD,Pelican Sunstream 100 Kayak,199.99,Standard Class
3,DEBIT,6,4,-41.89,175.99,Late delivery,1,48,Water Sports,Littleton,...,1,199.99,175.99,-41.89,East of USA,Nueva York,COMPLETE,Pelican Sunstream 100 Kayak,199.99,Standard Class
4,DEBIT,6,4,10.00,40.00,Late delivery,1,24,Women's Apparel,Littleton,...,1,50.00,40.00,10.00,East of USA,Nueva York,COMPLETE,Nike Men's Dri-FIT Victory Golf Polo,50.00,Standard Class


In [3]:
df["Delay_Gap"] = df["Days for shipping (real)"] - df["Days for shipment (scheduled)"]
df["Delay_Gap"].describe()

count    180519.000000
mean          0.565807
std           1.490966
min          -2.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           4.000000
Name: Delay_Gap, dtype: float64

In [4]:
def classify_delivery(gap):
    if gap == 0:
        return "On-time"
    elif gap > 0:
        return "Delayed"
    else:
        return "Early"

df["Delivery_Class"] = df["Delay_Gap"].apply(classify_delivery)
df["Delivery_Class"].value_counts()

Delivery_Class
Delayed    103400
Early       43366
On-time     33753
Name: count, dtype: int64

In [ ]:
#Calculate on-time delivery percentage

In [6]:
on_time_pct = (df["Delivery_Class"] == "On-time").mean() * 100
print(f"On-Time Delivery Rate: {on_time_pct:.2f}%")

On-Time Delivery Rate: 18.70%


In [ ]:
#Late Delivery Risk Distribution

In [7]:
risk_counts = df["Late_delivery_risk"].value_counts()
risk_pct = df["Late_delivery_risk"].value_counts(normalize=True) * 100

print(risk_counts)
print(risk_pct)

Late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64
Late_delivery_risk
1    54.829132
0    45.170868
Name: proportion, dtype: float64


In [8]:
 #Baseline logistics performance

In [9]:
baseline_summary = {
    "Total Orders": len(df),
    "On-Time Delivery Rate (%)": round((df["Delivery_Class"] == "On-time").mean() * 100, 2),
    "Delayed Rate (%)": round((df["Delivery_Class"] == "Delayed").mean() * 100, 2),
    "Early Rate (%)": round((df["Delivery_Class"] == "Early").mean() * 100, 2),
    "Late Delivery Risk Ratio (%)": round(df["Late_delivery_risk"].mean() * 100, 2),
    "Average Delay Gap (days)": round(df["Delay_Gap"].mean(), 2),
}

baseline_df = pd.DataFrame(list(baseline_summary.items()), columns=["Metric", "Value"])
baseline_df

,Metric,Value
0,Total Orders,180519.00
1,On-Time Delivery Rate (%),18.70
2,Delayed Rate (%),57.28
3,Early Rate (%),24.02
4,Late Delivery Risk Ratio (%),54.83
5,Average Delay Gap (days),0.57


In [10]:
#Shipping Mode Efficiency Analysis

In [11]:
mode_efficiency = df.groupby("Shipping Mode")["Delay_Gap"].mean().sort_values(ascending=False)
print(mode_efficiency)

Shipping Mode
Second Class      1.990828
First Class       1.000000
Same Day          0.478279
Standard Class   -0.004093
Name: Delay_Gap, dtype: float64


In [12]:
sla_compliance = df.groupby("Shipping Mode")["Late_delivery_risk"].apply(lambda x: (x == 0).mean() * 100).sort_values(ascending=False)
print(sla_compliance)

Shipping Mode
Standard Class    61.928317
Same Day          54.256958
Second Class      23.367219
First Class        4.677501
Name: Late_delivery_risk, dtype: float64


In [13]:
status_delay = df.groupby("Delivery Status")["Delay_Gap"].mean().sort_values(ascending=False)
print(status_delay)

Delivery Status
Late delivery        1.618184
Shipping canceled    0.572737
Shipping on time     0.000000
Advance shipping    -1.501851
Name: Delay_Gap, dtype: float64


In [14]:
#Regional & Market Diagnostics

In [15]:
regional_risk = df.groupby("Order Region")["Late_delivery_risk"].mean().sort_values(ascending=False) * 100
print(regional_risk)

Order Region
Central Africa     57.960644
South Asia         56.266977
East Africa        55.939525
Western Europe     55.848611
South of  USA      55.772559
Eastern Europe     55.663265
East of USA        55.661605
Southeast Asia     55.529930
Central Asia       55.334539
West Asia          55.283741
US Center          55.240360
Central America    54.754596
North Africa       54.517327
Southern Europe    54.384477
Eastern Asia       54.326923
South America      54.308671
Northern Europe    54.044118
Oceania            54.020497
West of USA        53.959715
Southern Africa    53.327571
Caribbean          53.077663
West Africa        52.840909
Canada             48.800834
Name: Late_delivery_risk, dtype: float64


In [16]:
country_risk = df.groupby("Order Country")["Late_delivery_risk"].mean().sort_values(ascending=False) * 100
country_risk.head(10)

Order Country
República de Gambia    100.0
Armenia                100.0
Sáhara Occidental      100.0
Bután                  100.0
Sudán del Sur          100.0
Luxemburgo             100.0
Laos                   100.0
Suazilandia            100.0
Eritrea                100.0
Guinea Ecuatorial      100.0
Name: Late_delivery_risk, dtype: float64

In [17]:
market_risk = df.groupby("Market")["Late_delivery_risk"].mean().sort_values(ascending=False) * 100
print(market_risk)

Market
Europe          55.207753
Pacific Asia    55.046049
USCA            54.800574
Africa          54.589289
LATAM           54.355158
Name: Late_delivery_risk, dtype: float64


In [18]:
regional_delay_days = df.groupby("Order Region")["Delay_Gap"].mean().sort_values(ascending=False)
print(regional_delay_days)

Order Region
Central Asia       0.645570
Central Africa     0.639833
South Asia         0.597465
Western Europe     0.597403
US Center          0.587226
East of USA        0.584816
South of  USA      0.579975
Eastern Europe     0.579847
East Africa        0.570734
West Asia          0.569479
Eastern Asia       0.566484
Central America    0.561942
Southeast Asia     0.558235
West of USA        0.557238
South America      0.556344
Oceania            0.556267
North Africa       0.552290
West Africa        0.550595
Northern Europe    0.546875
Caribbean          0.546526
Southern Europe    0.515640
Southern Africa    0.478825
Canada             0.391032
Name: Delay_Gap, dtype: float64


In [19]:
#Customer Segment Impact Analysis

In [20]:
segment_risk = df.groupby("Customer Segment")["Late_delivery_risk"].mean().sort_values(ascending=False) * 100
print(segment_risk)

Customer Segment
Home Office    55.070440
Consumer       54.808350
Corporate      54.722663
Name: Late_delivery_risk, dtype: float64


In [21]:
segment_delay = df.groupby("Customer Segment")["Delay_Gap"].mean().sort_values(ascending=False)
print(segment_delay)

Customer Segment
Home Office    0.577949
Consumer       0.564917
Corporate      0.560185
Name: Delay_Gap, dtype: float64


In [22]:
segment_mode = df.groupby(["Customer Segment", "Shipping Mode"])["Late_delivery_risk"].mean().unstack() * 100
segment_mode

Shipping Mode,First Class,Same Day,Second Class,Standard Class
Customer Segment,,,,
Consumer,95.148192,46.484071,76.118242,38.332857
Corporate,95.435885,46.184874,77.899543,37.364189
Home Office,95.629268,42.565056,75.897272,38.510771


In [23]:
analysis_summary = {
    "On-Time Delivery Rate (%)": round((df["Delivery_Class"] == "On-time").mean() * 100, 2),
    "Late Delivery Risk Ratio (%)": round(df["Late_delivery_risk"].mean() * 100, 2),
    "Avg Delay Gap (days)": round(df["Delay_Gap"].mean(), 2),
    "Worst Shipping Mode": df.groupby("Shipping Mode")["Delay_Gap"].mean().idxmax(),
    "Best Shipping Mode": df.groupby("Shipping Mode")["Delay_Gap"].mean().idxmin(),
    "Worst Region": df.groupby("Order Region")["Late_delivery_risk"].mean().idxmax(),
    "Best Region": df.groupby("Order Region")["Late_delivery_risk"].mean().idxmin(),
}
pd.DataFrame(list(analysis_summary.items()), columns=["Metric", "Value"])

,Metric,Value
0,On-Time Delivery Rate (%),18.7
1,Late Delivery Risk Ratio (%),54.83
2,Avg Delay Gap (days),0.57
3,Worst Shipping Mode,Second Class
4,Best Shipping Mode,Standard Class
5,Worst Region,Central Africa
6,Best Region,Canada


In [24]:
df.to_csv("analysis_ready_data.csv", index=False)
print("Saved! Shape:", df.shape)
print("Columns:", df.columns.tolist())

Saved! Shape: (180519, 42)
Columns: ['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Product Name', 'Product Price', 'Shipping Mode', 'Delay_Gap', 'Delivery_Class']
